# Azure AI-103 Labs — Unique Code Snippets Reference

A deduplicated reference of every **distinct** reusable code pattern found across the 21 lab scripts in `14-azure-ai103/Labs/`. Each pattern appears **once**, even though the same idiom (e.g. building a bearer-token credential, connecting to a Foundry agent) is repeated near-identically across several lab files.

Cells are NOT meant to run top-to-bottom as one program — each is an independent snippet lifted from a specific lab (noted above the cell). Fill in your own `.env` values and run the cells you need.

See [`README.md`](README.md) in this folder for the full per-file walkthrough these snippets were extracted from.


## 1. Shared setup utilities

Used at the top of almost every lab file.

**Clear the console cross-platform** — appears in nearly every lab (Labs 3, 6, 12–16).

In [1]:
import os

os.system('cls' if os.name == 'nt' else 'clear')


0

**Load configuration from `.env`** — the pattern behind every `os.getenv(...)` call in this folder (variable names differ per lab — see `README.md`'s env-var table).

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")   # substitute whichever variable name the lab you're working from uses


## 2. Authentication patterns

Every lab authenticates via your `az login` session (Entra ID) — never a hardcoded API key.

**Bearer token provider for an OpenAI-compatible client** (sync) — Lab 3 (all 4 variants), Lab 14 (both files).

In [3]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
)


**Bearer token provider, async version** (`azure.identity.aio`) — Lab 3 `chat-async.py`. The credential must be explicitly closed when you're done with it.

In [4]:
from azure.identity.aio import DefaultAzureCredential, get_bearer_token_provider

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential, "https://ai.azure.com/.default"
)

# ... use the credential ...

await credential.close()


**Plain `DefaultAzureCredential` for a Foundry project client** (no token provider needed — the SDK client accepts the credential object directly) — Labs 6, 7, 8, 9, 12, 13, 16.

In [5]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()


**`DefaultAzureCredential` with specific auth methods excluded** — Lab 9. Skips environment-variable and managed-identity auth, forcing your `az login` (Azure CLI) session to be used.

In [6]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential(
    exclude_environment_credential=True,
    exclude_managed_identity_credential=True
)


**`AzureCliCredential`** — an alternative to `DefaultAzureCredential` that goes straight to your Azure CLI session, used by the Microsoft Agent Framework labs (10, 11) and the async VoiceLive lab (17).

In [7]:
from azure.identity import AzureCliCredential

credential = AzureCliCredential()


## 3. Generative AI — Azure OpenAI clients & chat calls (Lab 3–4)

**`OpenAI` client pointed at an Azure endpoint** (sync) — the OpenAI-compatible way to reach Azure OpenAI through Foundry. Lab 3 (`chat-app-chatcompletion.py`, `chat-app-responseapi.py`, `chat-app-responseapi-stream.py`).

In [8]:
import sys
sys.path.insert(0, "..")
from azure_config import config

from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

azure_openai_endpoint = config.openai_endpoint
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
)

openai_client = OpenAI(
    base_url=azure_openai_endpoint,
    api_key=token_provider
)


**`AsyncOpenAI` client** (async equivalent of the above) — Lab 3 `chat-async.py`.

In [9]:
import sys
sys.path.insert(0, "..")
from azure_config import config

from openai import AsyncOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

azure_openai_endpoint = config.openai_endpoint
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
)

async_client = AsyncOpenAI(
    base_url=azure_openai_endpoint,
    api_key=token_provider
)


**`AzureOpenAI` client for audio models** (TTS / transcription) — Lab 14 (both files). Needs an explicit `api_version`, unlike the plain `OpenAI` client above.

In [10]:
import sys
sys.path.insert(0, "..")
from azure_config import config

from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

endpoint = config.openai_endpoint
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
)

client = AzureOpenAI(
    azure_endpoint=endpoint,
    azure_ad_token_provider=token_provider,
    api_version="2025-03-01-preview"
)


**Chat Completions API call** (the classic/older shape — you resend the whole `messages` list every turn) — Lab 3 `chat-app-chatcompletion.py`.

In [11]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")
openai_client = OpenAI(base_url=config.openai_endpoint, api_key=token_provider)
model_deployment = config.openai_deployment
input_text = "What is the capital of France?"   # stand-in for the interactive input() in the original lab script

completion = openai_client.chat.completions.create(
    model=model_deployment,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant that answers questions and provides information."},
        {"role": "user", "content": input_text}
    ]
)
print(completion.choices[0].message.content)


AuthenticationError: Error code: 401 - {'error': {'code': 'PermissionDenied', 'message': 'Principal does not have access to API/Operation.'}}

**Responses API call with conversation memory** (the newer shape — no message list, memory tracked server-side via `previous_response_id`) — Lab 3 `chat-app-responseapi.py`.

In [12]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")
openai_client = OpenAI(base_url=config.openai_endpoint, api_key=token_provider)
model_deployment = config.openai_deployment
input_text = "What is the capital of France?"   # stand-in for the interactive input() in the original lab script
last_response_id = None   # None on the first turn; set to response.id below for a real second turn

response = openai_client.responses.create(
    model=model_deployment,
    instructions="You are a helpful AI assistant that answers questions and provides information.",
    input=input_text,
    previous_response_id=last_response_id,
)
print(response.output_text)
last_response_id = response.id


AuthenticationError: Error code: 401 - {'error': {'code': 'PermissionDenied', 'message': 'Principal does not have access to API/Operation.'}}

**Responses API call, streamed token-by-token** — Lab 3 `chat-app-responseapi-stream.py`.

In [13]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")
openai_client = OpenAI(base_url=config.openai_endpoint, api_key=token_provider)
model_deployment = config.openai_deployment
input_text = "What is the capital of France?"   # stand-in for the interactive input() in the original lab script
last_response_id = None

stream = openai_client.responses.create(
    model=model_deployment,
    instructions="You are a helpful AI assistant that answers questions and provides information.",
    input=input_text,
    previous_response_id=last_response_id,
    stream=True
)
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="")
    elif event.type == "response.completed":
        last_response_id = event.response.id


AuthenticationError: Error code: 401 - {'error': {'code': 'PermissionDenied', 'message': 'Principal does not have access to API/Operation.'}}

**Responses API call, async/await** — Lab 3 `chat-async.py`.

In [14]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from openai import AsyncOpenAI
from azure.identity.aio import DefaultAzureCredential, get_bearer_token_provider

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
async_client = AsyncOpenAI(base_url=config.openai_endpoint, api_key=token_provider)
model_deployment = config.openai_deployment
input_text = "What is the capital of France?"   # stand-in for the interactive input() in the original lab script
last_response_id = None

response = await async_client.responses.create(
    model=model_deployment,
    instructions="You are a helpful AI assistant that answers questions and provides information.",
    input=input_text,
    previous_response_id=last_response_id
)
assistant_text = response.output_text
last_response_id = response.id
print(assistant_text)

await credential.close()


AuthenticationError: Error code: 401 - {'error': {'code': 'PermissionDenied', 'message': 'Principal does not have access to API/Operation.'}}

## 4. Foundry Agent Service — connecting & conversations (Lab 6–9, 12–13, 16)

**Connect to a Foundry project (plain-variable style)** — Labs 6, 9, 12, 13, 16.

In [15]:
import sys
sys.path.insert(0, "..")
from azure_config import config

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_endpoint = config.project_endpoint

project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()


**Connect to a Foundry project (context-manager style)** — closes the credential/clients automatically. Labs 7, 8.

In [16]:
import sys
sys.path.insert(0, "..")
from azure_config import config

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_endpoint = config.project_endpoint

with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=project_endpoint, credential=credential) as project_client,
    project_client.get_openai_client() as openai_client,
):
    print(f"Connected: {project_client}, {openai_client}")   # stand-in for "...  # use project_client / openai_client inside this block"


Connected: <azure.ai.projects._patch.AIProjectClient object at 0x112946ed0>, <openai.OpenAI object at 0x112986410>


**Look up an agent already created in the Foundry portal** (by name — this code never creates or edits the agent) — Labs 6, 9, 13, 16.

In [17]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(endpoint=config.project_endpoint, credential=DefaultAzureCredential())
agent_name = config.agent_name("IT-HelpDesk-Agent")   # a real agent already created in this project (see 11_azure_ai_foundry/05_hosted_agent)

agent = project_client.agents.get(agent_name=agent_name)
print(f"Connected to agent: {agent.name} (id: {agent.id})")


Connected to agent: cloudxeus-support-rag-agent (id: cloudxeus-support-rag-agent)


**Create a new agent version in code** (`PromptAgentDefinition`) — Labs 7, 8, 10 (Agent Framework equivalent shown separately below).

In [18]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(endpoint=config.project_endpoint, credential=DefaultAzureCredential())
model_deployment = config.model_deployment

# The other two tools referenced below (only event_tool is spelled out earlier in this
# catalog, in section 5) - defined here too so this cell is runnable standalone.
event_tool = FunctionTool(
    name="next_visible_event",
    description="Get the next visible event in a given location.",
    parameters={
        "type": "object",
        "properties": {"location": {"type": "string", "description": "continent, e.g. 'north_america'"}},
        "required": ["location"],
        "additionalProperties": False,
    },
    strict=True,
)
cost_tool = FunctionTool(
    name="calculate_observation_cost",
    description="Calculate the cost of an observation based on the telescope tier, number of hours, and priority level.",
    parameters={
        "type": "object",
        "properties": {
            "telescope_tier": {"type": "string", "description": "e.g. 'standard', 'advanced', 'premium'"},
            "hours": {"type": "number", "description": "number of hours"},
            "priority": {"type": "string", "description": "e.g. 'low', 'normal', 'high'"},
        },
        "required": ["telescope_tier", "hours", "priority"],
        "additionalProperties": False,
    },
    strict=True,
)
report_tool = FunctionTool(
    name="generate_observation_report",
    description="Generate a report summarizing an astronomical observation",
    parameters={
        "type": "object",
        "properties": {
            "event_name": {"type": "string"}, "location": {"type": "string"},
            "telescope_tier": {"type": "string"}, "hours": {"type": "number"},
            "priority": {"type": "string"}, "observer_name": {"type": "string"},
        },
        "required": ["event_name", "location", "telescope_tier", "hours", "priority", "observer_name"],
        "additionalProperties": False,
    },
    strict=True,
)

agent = project_client.agents.create_version(
    agent_name="astronomy-agent",
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions="You are an astronomy observations assistant...",
        tools=[event_tool, cost_tool, report_tool],   # omit `tools=` entirely for a plain, tool-less agent
    ),
)
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")


Agent created (id: astronomy-agent:1, name: astronomy-agent, version: 1)


**Delete an agent version** (cleanup, for agents your own script created) — Labs 7, 8.

In [19]:
# Meant to run right after the "create a new agent version" cell above (same
# create-then-delete pairing the original lab scripts do within one run) -
# `project_client` and `agent` come from that cell.
project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("Deleted agent.")


Deleted agent.


**Create a conversation** (server-side message-history container) — Labs 6–9, 12–13, 16.

In [20]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(endpoint=config.project_endpoint, credential=DefaultAzureCredential())
openai_client = project_client.get_openai_client()

conversation = openai_client.conversations.create()
# or, equivalently: openai_client.conversations.create(items=[])
print(f"Conversation created (id: {conversation.id})")


Conversation created (id: conv_9d06f8575cf8ee37005kGa6leWTckHbieCkiU7g3DkVRoM3Fuk)


**Add a user message to a conversation**

In [21]:
# Meant to run right after the "create a conversation" cell above - reuses its
# `openai_client` / `conversation`.
user_input = "What's the status of my VPN connection issue?"   # stand-in for interactive input()

openai_client.conversations.items.create(
    conversation_id=conversation.id,
    items=[{"type": "message", "role": "user", "content": user_input}],
)
print("Message added to conversation.")


Message added to conversation.


**Get an agent's reply, continuing an existing conversation** (`input=""` — the message was already added to the conversation above) — Labs 6, 9.

In [22]:
# Meant to run right after the two cells above (create a conversation, add a user
# message) - reuses their `openai_client` / `conversation`.
agent_name = config.agent_name("IT-HelpDesk-Agent")
agent = project_client.agents.get(agent_name=agent_name)

response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input=""
)
print(response.output_text)


I don't have that information in the available knowledge base.

Please contact support@cloudxeus.com so our team can look up your specific ticket and provide the current status.


**Get an agent's reply, one-shot with no conversation object** (no memory across calls) — Labs 13, 16.

In [23]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(endpoint=config.project_endpoint, credential=DefaultAzureCredential())
openai_client = project_client.get_openai_client()
agent_name = config.agent_name("IT-HelpDesk-Agent")
prompt = "How do I reset my password?"

response = openai_client.responses.create(
    input=[{"role": "user", "content": prompt}],
    extra_body={"agent_reference": {"name": agent_name, "type": "agent_reference"}},
)
print(f"{agent_name}: {response.output_text}")


cloudxeus-support-rag-agent: I don't have that information in the available knowledge base.

Please contact support@cloudxeus.com so our team can help you reset your password.


## 5. Function calling / tools (Lab 7–8)

**Define a `FunctionTool`** (JSON-schema description of a Python function so the model knows it exists) — Lab 7 (three of these, one per tool — shown here is the representative shape; parameters differ per tool).

In [24]:
from azure.ai.projects.models import FunctionTool

event_tool = FunctionTool(
    name="next_visible_event",
    description="Get the next visible event in a given location.",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "continent to find the next visible event in (e.g. 'north_america', 'south_america', 'australia')",
            },
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    strict=True,
)


**Process `function_call` items and dispatch by name (hardcoded if/elif)** — Lab 7 `agent.py`.

In [25]:
import json
from types import SimpleNamespace
from openai.types.responses.response_input_param import FunctionCallOutput

# Stand-ins for the real Lab 7 helpers (functions.py reads local data/ files not
# included in this repo) - same signatures, canned output, so this dispatch
# pattern is demonstrable without that data folder.
def next_visible_event(location: str) -> str:
    return json.dumps({"event": "Total Solar Eclipse", "date": "04-08", "visible_from": [location]})

def calculate_observation_cost(telescope_tier: str, hours: float, priority: str) -> str:
    return json.dumps({"telescope_tier": telescope_tier, "hours": hours, "priority": priority, "total_cost": hours * 50.0})

def generate_observation_report(**kwargs) -> str:
    return json.dumps({"status": "Report generated", "file": "report.txt"})

# A `response` normally comes from a live agent call with function tools (see
# section 4) - simulated here so this dispatch loop is demonstrable in isolation.
response = SimpleNamespace(output=[
    SimpleNamespace(type="function_call", name="next_visible_event",
                    arguments=json.dumps({"location": "north_america"}), call_id="call_1"),
])
input_list = []

for item in response.output:
    if item.type == "function_call":
        result = None
        if item.name == "next_visible_event":
            result = next_visible_event(**json.loads(item.arguments))
        elif item.name == "calculate_observation_cost":
            result = calculate_observation_cost(**json.loads(item.arguments))
        elif item.name == "generate_observation_report":
            result = generate_observation_report(**json.loads(item.arguments))

        input_list.append(
            FunctionCallOutput(
                type="function_call_output",
                call_id=item.call_id,
                output=result,
            )
        )

print(input_list)


[{'type': 'function_call_output', 'call_id': 'call_1', 'output': '{"event": "Total Solar Eclipse", "date": "04-08", "visible_from": ["north_america"]}'}]


**Process `function_call` items and dispatch via a lookup dict** (generalized — works for any number of dynamically-discovered tools) — Lab 8 `client.py`.

In [26]:
import json
from types import SimpleNamespace
from openai.types.responses.response_input_param import FunctionCallOutput

# Stand-in for Lab 8's MCP-backed tool functions (see section 6 for the real
# MCP wiring) - a plain async function returning a mock "MCP result" shape
# (an object with .content[0].text) so this dispatch loop is demonstrable
# without a running MCP server.
async def get_inventory_levels(**kwargs):
    return SimpleNamespace(content=[SimpleNamespace(text=json.dumps({"Moisturizer": 6, "Shampoo": 8}))])

functions_dict = {"get_inventory_levels": get_inventory_levels}

# A `response` normally comes from a live agent call with MCP-backed function
# tools - simulated here so this dispatch loop is demonstrable in isolation.
response = SimpleNamespace(output=[
    SimpleNamespace(type="function_call", name="get_inventory_levels", arguments="{}", call_id="call_1"),
])
input_list = []

for item in response.output:
    if item.type == "function_call":
        kwargs = json.loads(item.arguments)
        required_function = functions_dict.get(item.name)
        output = await required_function(**kwargs)

        input_list.append(
            FunctionCallOutput(
                type="function_call_output",
                call_id=item.call_id,
                output=output.content[0].text,
            )
        )

print(input_list)


[{'type': 'function_call_output', 'call_id': 'call_1', 'output': '{"Moisturizer": 6, "Shampoo": 8}'}]


**Send function call outputs back to the model to get its final answer**

In [27]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(endpoint=config.project_endpoint, credential=DefaultAzureCredential())
openai_client = project_client.get_openai_client()
agent_name = config.agent_name("IT-HelpDesk-Agent")
agent = project_client.agents.get(agent_name=agent_name)

response = openai_client.responses.create(
    input=[{"role": "user", "content": "Hello"}],
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)
input_list = []   # populated with FunctionCallOutput items by the dispatch loop in section 5 above

if input_list:
    response = openai_client.responses.create(
        input=input_list,
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )
print(f"AGENT: {response.output_text}")


AGENT: Hello! How can I help you today with CloudXeus Technology Services?

I don't have that information in the available knowledge base.  
For more details, please contact support@cloudxeus.com.


## 6. MCP — Model Context Protocol (Lab 8)

**Remote/hosted MCP tool with mandatory human approval** — Lab 8 `agent.py`.

In [28]:
from azure.ai.projects.models import MCPTool

mcp_tool = MCPTool(
    server_label="api-specs",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="always",
)


**Approve pending MCP tool calls in a loop until none remain**

In [29]:
from types import SimpleNamespace
from openai.types.responses.response_input_param import McpApprovalResponse

# `response` normally comes from a live agent call using an MCPTool with
# require_approval="always" (see the MCPTool cell above) - simulated here
# (one pending approval, then none) so this approval loop is demonstrable
# without a live agent round-trip.
response = SimpleNamespace(output=[
    SimpleNamespace(type="mcp_approval_request", id="approval_1"),
])

class _FakeResponses:
    def create(self, **kwargs):
        return SimpleNamespace(output=[], output_text="(approved - final answer would appear here)")

openai_client = SimpleNamespace(responses=_FakeResponses())
agent = SimpleNamespace(name="MyAgent")

while True:
    input_list = []
    for item in response.output:
        if item.type == "mcp_approval_request":
            input_list.append(
                McpApprovalResponse(
                    type="mcp_approval_response",
                    approve=True,
                    approval_request_id=item.id,
                )
            )

    if not input_list:
        break

    response = openai_client.responses.create(
        input=input_list,
        previous_response_id=getattr(response, "id", None),
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )

print(response.output_text)


(approved - final answer would appear here)


**Build a local MCP server with `fastmcp`** — Lab 8 `server.py`.

In [30]:
# ⚠️ Do not run this cell inside this notebook - mcp.run() blocks forever,
# listening on stdio, which would hang the kernel. It's saved as a real,
# runnable file alongside this notebook at mcp_demo_server.py (identical code)
# so the stdio-client cell in this same section can launch it as a subprocess.
from fastmcp import FastMCP

mcp = FastMCP(name="Inventory")

@mcp.tool()
def get_inventory_levels() -> dict:
    """Returns current inventory for all products."""
    return {"Moisturizer": 6, "Shampoo": 8}

mcp.run(show_banner=False)


RuntimeError: Already running asyncio in this thread

**Connect to a local MCP server as a subprocess (stdio transport)** — Lab 8 `client.py`.

In [31]:
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

exit_stack = AsyncExitStack()

server_params = StdioServerParameters(command="python", args=["mcp_demo_server.py"], env=None)   # see mcp_demo_server.py alongside this notebook

stdio_transport = await exit_stack.enter_async_context(stdio_client(server_params))
stdio, write = stdio_transport

session = await exit_stack.enter_async_context(ClientSession(stdio, write))
await session.initialize()

response = await session.list_tools()
tools = response.tools
print("Connected to server with tools:", [tool.name for tool in tools])

# `anyio`'s cancel scopes are task-scoped, and each Jupyter cell runs as its own
# asyncio task - so exit_stack must be entered AND closed within this same cell,
# not deferred to the next one (that raises "Attempted to exit cancel scope in a
# different task than it was entered in"). `session`/`tools` are still usable as
# plain Python objects in the next cell - they're just no longer connected.
await exit_stack.aclose()


Connected to server with tools: ['get_inventory_levels']


**Wrap MCP tools as Foundry `FunctionTool`s dynamically** (bridges any MCP server's tools into an agent) — Lab 8 `client.py`.

In [32]:
# Meant to run right after the cell above (reuses its `tools`; the MCP session
# itself is already closed by then, but building the FunctionTool wrappers below
# doesn't require an open connection).
from azure.ai.projects.models import FunctionTool

def make_tool_func(tool_name):
    async def tool_func(**kwargs):
        return await session.call_tool(tool_name, kwargs)
    tool_func.__name__ = tool_name
    return tool_func

functions_dict = {tool.name: make_tool_func(tool.name) for tool in tools}

mcp_function_tools = [
    FunctionTool(
        name=tool.name,
        description=tool.description,
        parameters={"type": "object", "properties": {}, "additionalProperties": False},
        strict=True,
    )
    for tool in tools
]
print([t.name for t in mcp_function_tools])


['get_inventory_levels']


## 7. Microsoft Agent Framework (Lab 10–11)

**Create a Foundry-backed chat client**

In [33]:
import sys
sys.path.insert(0, "..")
import os
from azure_config import config

try:
    from agent_framework.foundry import FoundryChatClient
    from azure.identity import AzureCliCredential

    client = FoundryChatClient(
        project_endpoint=config.project_endpoint,
        model=config.model_deployment,
        credential=AzureCliCredential()
    )
    print(f"Created: {client}")
except ImportError as ex:
    print(f"⚠️  agent_framework not installed ({ex}) - run `pip install agent-framework-foundry` to try this cell for real.")


⚠️  agent_framework not installed (No module named 'agent_framework.foundry') - run `pip install agent-framework-foundry` to try this cell for real.


**Define an agent tool with the `@tool` decorator** (schema is derived automatically from type hints — no hand-written JSON schema) — Lab 10.

In [34]:
try:
    from agent_framework import tool
    from pydantic import Field
    from typing import Annotated

    @tool(approval_mode="never_require")
    def submit_claim(
        to: Annotated[str, Field(description="Who to send the email to")],
        subject: Annotated[str, Field(description="The subject of the email.")],
        body: Annotated[str, Field(description="The text body of the email.")]):
            print("\nTo:", to)
            print("Subject:", subject)
            print(body, "\n")
    print("submit_claim tool defined.")
except ImportError as ex:
    print(f"⚠️  agent_framework not installed ({ex}) - run `pip install agent-framework-foundry` to try this cell for real.")


⚠️  agent_framework not installed (cannot import name 'tool' from 'agent_framework' (unknown location)) - run `pip install agent-framework-foundry` to try this cell for real.


**Create and run an agent as an async context manager** — Lab 10.

In [35]:
# Meant to run right after the two cells above (reuses their `client` / `submit_claim`).
try:
    from agent_framework import Agent

    async with (
        Agent(
            client=client,
            name="ExpenseClaimAgent",
            instructions="You are an AI assistant for expense claim submission...",
            tools=[submit_claim],
        ) as agent,
    ):
        response = await agent.run(["some prompt"])
        print(response)
except ImportError as ex:
    print(f"⚠️  agent_framework not installed ({ex}) - run `pip install agent-framework-foundry` to try this cell for real.")


⚠️  agent_framework not installed (cannot import name 'Agent' from 'agent_framework' (unknown location)) - run `pip install agent-framework-foundry` to try this cell for real.


**Create an agent with the `as_agent()` shorthand** (no tools needed) — Lab 11.

In [36]:
import sys
sys.path.insert(0, "..")
from azure_config import config

try:
    from agent_framework.foundry import FoundryChatClient
    from azure.identity import AzureCliCredential

    chat_client = FoundryChatClient(
        credential=AzureCliCredential(),
        project_endpoint=config.project_endpoint,
        model=config.model_deployment,
    )

    summarizer_agent = chat_client.as_agent(
        name="summarizer",
        instructions="Summarize the customer's feedback in one short sentence.",
    )
    print(f"Created: {summarizer_agent}")
except ImportError as ex:
    print(f"⚠️  agent_framework not installed ({ex}) - run `pip install agent-framework-foundry` to try this cell for real.")


⚠️  agent_framework not installed (No module named 'agent_framework.foundry') - run `pip install agent-framework-foundry` to try this cell for real.


**Sequential multi-agent orchestration** (output of one agent feeds the next) — Lab 11.

In [37]:
# Meant to run right after the cell above (reuses its `chat_client`).
try:
    from agent_framework.orchestrations import SequentialBuilder

    classifier_agent = chat_client.as_agent(name="classifier", instructions="Classify the feedback as Positive, Negative, or Feature request.")
    action_agent = chat_client.as_agent(name="action", instructions="Suggest the next action in one short sentence.")

    workflow = SequentialBuilder(
        participants=[summarizer_agent, classifier_agent, action_agent],
        output_from="all",
    ).build()

    result = await workflow.run("Customer feedback: ...")
    outputs = result.get_outputs()
    print(outputs)
except ImportError as ex:
    print(f"⚠️  agent_framework not installed ({ex}) - run `pip install agent-framework-foundry` to try this cell for real.")
except NameError as ex:
    print(f"⚠️  {ex} - run the cell above first (agent_framework wasn't available, so `chat_client`/`summarizer_agent` weren't created).")


⚠️  agent_framework not installed (No module named 'agent_framework.orchestrations') - run `pip install agent-framework-foundry` to try this cell for real.


## 8. Azure AI Language — Text Analytics (Lab 12)

**Create a `TextAnalyticsClient`**

In [38]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.ai.textanalytics import TextAnalyticsClient
from azure.identity import DefaultAzureCredential

foundry_endpoint = config.language_endpoint   # raises MissingCredentialError until AZURE_LANGUAGE_ENDPOINT is set in .env (no Language resource provisioned yet in this project)

ai_client = TextAnalyticsClient(endpoint=foundry_endpoint, credential=DefaultAzureCredential())


MissingCredentialError: 'AZURE_LANGUAGE_ENDPOINT' is not set. Add it to the repo-root .env file (see 14-azure-ai103/README.md for the full variable list).

**Detect language, extract entities, and redact PII**

In [39]:
# Meant to run right after the cell above (reuses its `ai_client`).
text = "My name is John Smith and I love the new dashboard feature, though it crashed once on Tuesday. Call me at 555-0142."

detected_language = ai_client.detect_language(documents=[text])[0]
print(detected_language.primary_language.name)

entities = ai_client.recognize_entities(documents=[text])[0].entities
for entity in entities:
    print(f"{entity.text} ({entity.category})")

pii_result = ai_client.recognize_pii_entities(documents=[text])[0]
for pii_entity in pii_result.entities:
    print(f"{pii_entity.text} ({pii_entity.category})")
print(pii_result.redacted_text)


NameError: name 'ai_client' is not defined

## 9. Azure AI Speech & Azure OpenAI audio models (Lab 14–16)

**Text-to-speech via an Azure OpenAI audio model, streamed to a file** — Lab 14 `generate-speech.py`.

In [40]:
import sys
sys.path.insert(0, "..")
from pathlib import Path
from azure_config import config
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")
client = AzureOpenAI(azure_endpoint=config.openai_endpoint, azure_ad_token_provider=token_provider, api_version="2025-03-01-preview")
model_deployment = config.audio_deployment   # raises MissingCredentialError until AZURE_OPENAI_AUDIO_DEPLOYMENT is set in .env (no audio deployment provisioned yet)
speech_file_path = Path("speech.mp3")

with client.audio.speech.with_streaming_response.create(
    model=model_deployment,
    voice="alloy",
    input="My voice is my passport!",
    instructions="Speak in a serious tone.",
) as response:
    response.stream_to_file(speech_file_path)


MissingCredentialError: 'AZURE_OPENAI_AUDIO_DEPLOYMENT' is not set. Add it to the repo-root .env file (see 14-azure-ai103/README.md for the full variable list).

import sys
sys.path.insert(0, "..")
from azure_config import config
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")
client = AzureOpenAI(azure_endpoint=config.openai_endpoint, azure_ad_token_provider=token_provider, api_version="2025-03-01-preview")
model_deployment = config.audio_deployment   # raises MissingCredentialError until AZURE_OPENAI_AUDIO_DEPLOYMENT is set in .env
file_path = "speech.mp3"   # produced by the TTS cell above - swap in your own .wav/.mp3 to transcribe something else

audio_file = open(file_path, "rb")
transcription = client.audio.transcriptions.create(
    model=model_deployment,
    file=audio_file,
    response_format="text"
)
print(transcription)


In [41]:
audio_file = open(file_path, "rb")
transcription = client.audio.transcriptions.create(
    model=model_deployment,
    file=audio_file,
    response_format="text"
)
print(transcription)


NameError: name 'file_path' is not defined

**`SpeechConfig` authenticated via Entra ID** (dedicated Azure AI Speech SDK, no subscription key) — Lab 15.

In [42]:
import sys
sys.path.insert(0, "..")
from azure_config import config
import azure.cognitiveservices.speech as speech_sdk
from azure.identity import DefaultAzureCredential

foundry_endpoint = config.speech_endpoint   # raises MissingCredentialError until AZURE_SPEECH_ENDPOINT is set in .env (no Speech resource provisioned yet)

speech_config = speech_sdk.SpeechConfig(
    token_credential=DefaultAzureCredential(),
    endpoint=foundry_endpoint
)


MissingCredentialError: 'AZURE_SPEECH_ENDPOINT' is not set. Add it to the repo-root .env file (see 14-azure-ai103/README.md for the full variable list).

**Text-to-speech to a file with the dedicated Speech SDK** — Lab 15 `record_greeting()`.

In [43]:
# Meant to run right after the cell above (reuses its `speech_config`).
greeting_message = "Hi, you've reached the front desk. Please leave a message after the tone."

output_file = "greeting.wav"
audio_config = speech_sdk.audio.AudioOutputConfig(filename=output_file)
speech_config.speech_synthesis_voice_name = "en-US-Serena:DragonHDLatestNeural"

speech_synthesizer = speech_sdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_config)
result = speech_synthesizer.speak_text_async(greeting_message).get()

if result.reason == speech_sdk.ResultReason.SynthesizingAudioCompleted:
    print(f"Greeting recorded and saved to {output_file}")


NameError: name 'speech_config' is not defined

**Transcribe an audio file with the dedicated Speech SDK** — Lab 15 `transcribe_messages()`.

In [44]:
# Meant to run right after the two cells above - transcribes the greeting.wav
# they just synthesized (reuses `speech_config`).
file_path = "greeting.wav"

audio_config = speech_sdk.audio.AudioConfig(filename=file_path)
speech_recognizer = speech_sdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)
result = speech_recognizer.recognize_once_async().get()

if result.reason == speech_sdk.ResultReason.RecognizedSpeech:
    print(f"Transcription: {result.text}")


NameError: name 'speech_config' is not defined

## 10. Azure AI VoiceLive — real-time voice agent (Lab 17)

**Connect to VoiceLive and configure a real-time session**

In [45]:
import sys
sys.path.insert(0, "..")
from azure_config import config

try:
    from azure.ai.voicelive.aio import connect
    from azure.ai.voicelive.models import AgentConfig, RequestSession, Modality, InputAudioFormat, OutputAudioFormat, AzureSemanticVadMultilingual, AudioEchoCancellation, AudioNoiseReduction
    from azure.identity.aio import AzureCliCredential

    endpoint = config.voicelive_endpoint          # raises MissingCredentialError until AZURE_VOICELIVE_ENDPOINT is set
    agent_name = config.voicelive_agent_id
    project_name = config.voicelive_project_name
    credential = AzureCliCredential()
    agent_config = AgentConfig({"agent_name": agent_name, "project_name": project_name})

    async with connect(
        endpoint=endpoint,
        credential=credential,
        api_version="2026-01-01-preview",
        agent_config=agent_config
    ) as connection:
        session_config = RequestSession(
            modalities=[Modality.TEXT, Modality.AUDIO],
            input_audio_format=InputAudioFormat.PCM16,
            output_audio_format=OutputAudioFormat.PCM16,
            turn_detection=AzureSemanticVadMultilingual(),
            input_audio_echo_cancellation=AudioEchoCancellation(),
            input_audio_noise_reduction=AudioNoiseReduction(type="azure_deep_noise_suppression"),
        )
        await connection.session.update(session=session_config)
except ImportError as ex:
    print(f"⚠️  azure-ai-voicelive not installed ({ex}) - run `pip install azure-ai-voicelive` to try this cell for real.")


⚠️  azure-ai-voicelive not installed (No module named 'azure.ai.voicelive') - run `pip install azure-ai-voicelive` to try this cell for real.


**Process streamed server events** — the main real-time event loop.

In [46]:
# Meant to run right after the cell above (reuses its `connection`).
try:
    from azure.ai.voicelive.models import ServerEventType

    def start_microphone_capture(): print("(would start microphone capture here)")
    def queue_audio_for_playback(delta): pass
    def clear_playback_queue(): pass

    async for event in connection:
        if event.type == ServerEventType.SESSION_UPDATED:
            start_microphone_capture()
        elif event.type == ServerEventType.RESPONSE_AUDIO_DELTA:
            queue_audio_for_playback(event.delta)
        elif event.type == ServerEventType.INPUT_AUDIO_BUFFER_SPEECH_STARTED:
            clear_playback_queue()   # barge-in: stop the agent talking over the user
        elif event.type == ServerEventType.ERROR:
            print(f"Error: {event.error.message}")
except ImportError as ex:
    print(f"⚠️  azure-ai-voicelive not installed ({ex}) - run `pip install azure-ai-voicelive` to try this cell for real.")
except NameError as ex:
    print(f"⚠️  {ex} - run the cell above first (voicelive wasn't available, so `connection` wasn't created).")


⚠️  azure-ai-voicelive not installed (No module named 'azure.ai.voicelive') - run `pip install azure-ai-voicelive` to try this cell for real.


**Capture microphone audio and stream it to VoiceLive** (PyAudio input callback).

In [47]:
import base64
import asyncio

try:
    import pyaudio

    event_loop = asyncio.get_event_loop()

    def capture_callback(in_data, frame_count, time_info, status):
        audio_base64 = base64.b64encode(in_data).decode("utf-8")
        asyncio.run_coroutine_threadsafe(
            connection.input_audio_buffer.append(audio=audio_base64),
            event_loop
        )
        return (None, pyaudio.paContinue)

    audio = pyaudio.PyAudio()
    input_stream = audio.open(
        format=pyaudio.paInt16, channels=1, rate=24000,
        input=True, frames_per_buffer=1200,
        stream_callback=capture_callback
    )
    print("Microphone stream opened.")
except ImportError as ex:
    print(f"⚠️  pyaudio not installed ({ex}) - on macOS: `brew install portaudio && pip install pyaudio`.")
except NameError as ex:
    print(f"⚠️  {ex} - run the VoiceLive connect cell above first (needs a live `connection`).")


⚠️  pyaudio not installed (No module named 'pyaudio') - on macOS: `brew install portaudio && pip install pyaudio`.


**Play received audio through speakers, buffered via a queue** (PyAudio output callback).

In [48]:
import queue

try:
    import pyaudio

    playback_queue = queue.Queue()
    remaining = bytes()

    def playback_callback(in_data, frame_count, time_info, status):
        global remaining
        bytes_needed = frame_count * pyaudio.get_sample_size(pyaudio.paInt16)
        output = remaining[:bytes_needed]
        remaining = remaining[bytes_needed:]

        while len(output) < bytes_needed:
            try:
                audio_data = playback_queue.get_nowait()
                if audio_data is None:
                    break
                output += audio_data
            except queue.Empty:
                output += bytes(bytes_needed - len(output))  # pad with silence
                break

        return (output, pyaudio.paContinue)

    print("playback_callback defined.")
except ImportError as ex:
    print(f"⚠️  pyaudio not installed ({ex}) - on macOS: `brew install portaudio && pip install pyaudio`.")


⚠️  pyaudio not installed (No module named 'pyaudio') - on macOS: `brew install portaudio && pip install pyaudio`.


## 11. File / citation handling helpers (Lab 6)

Helpers for saving files an agent's code-interpreter tool generates or cites, since the response only contains a *reference* to sandboxed files, not the bytes themselves.

**Save binary/base64 data to a uniquely-named local file**

In [49]:
import base64
from pathlib import Path

OUTPUT_DIR = Path("agent_outputs")

def get_output_path(filename):
    OUTPUT_DIR.mkdir(exist_ok=True)
    file_name = Path(filename).name
    stem, suffix = Path(file_name).stem or "output", Path(file_name).suffix
    output_path = OUTPUT_DIR / file_name
    counter = 1
    while output_path.exists():
        output_path = OUTPUT_DIR / f"{stem}_{counter}{suffix}"
        counter += 1
    return output_path

def save_bytes(file_bytes, filename):
    output_path = get_output_path(filename)
    with open(output_path, "wb") as f:
        f.write(file_bytes)
    return output_path

def save_image(image_data, filename):
    return save_bytes(base64.b64decode(image_data), filename)


**Download a file cited by the agent's code-interpreter sandbox**

In [50]:
# References `save_bytes` from the cell above (only matters once this function is
# actually called - defining it here doesn't require save_bytes to exist yet).
def download_container_file(openai_client, annotation, downloaded_files):
    cache_key = (annotation.container_id, annotation.file_id)
    if cache_key in downloaded_files:
        return downloaded_files[cache_key]

    file_content = openai_client.containers.files.content.retrieve(
        file_id=annotation.file_id,
        container_id=annotation.container_id,
    )
    output_path = save_bytes(file_content.read(), annotation.filename or f"{annotation.file_id}.bin")
    downloaded_files[cache_key] = output_path
    return output_path


## 12. Prompt engineering & LLM-as-judge evaluation (Lab 5)

New in the "fill all missing" pass — Labs 18–28 close AI-103 gaps that had no hands-on lab coverage before. See `README.md`'s "AI-103 exam coverage" table for which of these are proven vs. best-effort/preview.

**Few-shot prompt** — worked examples baked into the prompt for a consistent output format, instead of a bare zero-shot question.

In [51]:
few_shot_prompt = """Classify the sentiment of a product review as exactly one word: Positive, Negative, or Mixed.

Example 1:
Review: "This blender is amazing, it crushes ice in seconds!"
Sentiment: Positive

Example 2:
Review: "The battery died after two days and support never replied."
Sentiment: Negative

Now classify this review:
Review: "{review_text}"
Sentiment:"""


**LLM-as-judge: draft → critique → conditional regenerate** — the real pattern this repo uses for "evaluation" (no `azure-ai-evaluation` SDK anywhere in this repo).

In [52]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(endpoint=config.project_endpoint, credential=DefaultAzureCredential())
openai_client = project_client.get_openai_client()
model_deployment = config.model_deployment
question = "What are the health benefits of drinking green tea?"

# Pass 1: draft answer
draft_response = openai_client.responses.create(model=model_deployment, input=question)
answer = draft_response.output_text

# Pass 2: judge call - same client, a grading prompt, a strict output convention
critique_response = openai_client.responses.create(
    model=model_deployment,
    input=(
        f"You are reviewing an AI assistant's answer for completeness.\n\n"
        f"Question: {question}\n\nAnswer: {answer}\n\n"
        "Reply with only COMPLETE or MISSING."
    ),
)
critique = critique_response.output_text

# Pass 3: only regenerate if the judge actually flagged a problem
if "MISSING" in critique.upper():
    improved = openai_client.responses.create(
        model=model_deployment,
        input=f"Answer this question completely, addressing every part: {question}",
    )
    answer = improved.output_text

print(answer)


Green tea has been studied quite extensively, and several health benefits are reasonably well-supported:

1. **Rich in antioxidants**  
   - Contains catechins (especially EGCG), which help reduce oxidative stress and cellular damage.  
   - May contribute to lower inflammation levels.

2. **Heart and vascular health**  
   - Regular consumption is associated with modest reductions in LDL (“bad”) cholesterol and triglycerides.  
   - Some studies show reduced risk of heart disease and stroke, especially in people who drink several cups per day.

3. **Metabolism and weight management**  
   - Catechins and caffeine together can slightly increase metabolic rate and fat oxidation.  
   - Effects are modest: useful as a support to diet and exercise, not a standalone weight-loss solution.

4. **Blood sugar and diabetes risk**  
   - May improve insulin sensitivity and help lower fasting blood glucose slightly.  
   - Habitual intake is linked with a lower risk of type 2 diabetes in some pop

## 13. Multi-agent delegation — `ConnectedAgentTool` (Lab 18)

Different from section 7's `SequentialBuilder` (pipes agents in sequence) — here ONE orchestrator agent calls other already-created agents AS TOOLS, on demand, in whatever order the conversation needs.

**Create specialist agents, then wrap each as a `ConnectedAgentTool`** — note `AgentsClient` (not `AIProjectClient`) and the older thread/run model.

In [53]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import ConnectedAgentTool
from azure.identity import DefaultAzureCredential

project_endpoint = config.project_endpoint
model_deployment = config.model_deployment

agents_client = AgentsClient(endpoint=project_endpoint, credential=DefaultAzureCredential())

specialist_agent = agents_client.create_agent(
    model=model_deployment, name="availability-checker", instructions="...",
)

specialist_tool = ConnectedAgentTool(
    id=specialist_agent.id,
    name="check_availability",
    description="Check whether a room type is available for given dates.",
)

# .definitions (plural, a LIST) - not the tool object itself - goes into tools=
orchestrator = agents_client.create_agent(
    model=model_deployment,
    name="concierge",
    instructions="Use your connected agents to answer the guest, combining their replies.",
    tools=specialist_tool.definitions,
)
print(f"Created: {orchestrator.name}, {specialist_agent.name}")


Created: concierge, availability-checker


**Run a thread against the orchestrator** — delegation to connected agents happens automatically inside `create_and_process()`, no manual function-call loop needed.

In [54]:
# Meant to run right after the cell above (reuses its `agents_client` / `orchestrator` / `specialist_agent`).
user_question = "I'd like a Deluxe room for 3 nights - is it available?"

thread = agents_client.threads.create()
agents_client.messages.create(thread_id=thread.id, role="user", content=user_question)
agents_client.runs.create_and_process(thread_id=thread.id, agent_id=orchestrator.id)

reply = agents_client.messages.get_last_message_text_by_role(thread_id=thread.id, role="assistant")
print(reply.text.value if reply else "(no reply)")

agents_client.delete_agent(orchestrator.id)
agents_client.delete_agent(specialist_agent.id)


(no reply)


## 14. Preview / best-effort patterns (Lab 19–21, 27)

⚠️ None of the class/method names below are confirmed against a real, current SDK — each script wraps its guessed import/call so it explains the concept instead of crashing uninformatively if the surface doesn't match. See each lab's file header for full caveats.

**Foundry IQ (shared knowledge platform)** — a knowledge source registered once, referenced by many agents (Lab 19). `KnowledgeTool` is a guessed class name, modeled on the real `MCPTool`/`FunctionTool` shape.

In [55]:
import sys
sys.path.insert(0, "..")
from azure_config import config

try:
    from azure.ai.projects.models import KnowledgeTool  # ⚠️ guessed — not confirmed to exist
    knowledge_source_name = config.knowledge_source_name   # raises MissingCredentialError until AZURE_AI_KNOWLEDGE_SOURCE_NAME is set
    knowledge_tool = KnowledgeTool(knowledge_source_name=knowledge_source_name)
    # Any number of agents can attach this SAME tool and all ground/cite consistently
except ImportError:
    print("KnowledgeTool not available on this SDK version - check current Foundry IQ docs.")


KnowledgeTool not available on this SDK version - check current Foundry IQ docs.


**A2A Agent Card** — the real, public discovery-document shape A2A agents publish (unlike the SDK class names elsewhere in this section, this JSON shape IS documented by the open A2A spec) — Lab 21.

In [56]:
agent_card = {
    "name": "astronomy-agent",
    "description": "Looks up astronomical events and calculates telescope rental costs.",
    "url": "https://example.com/agents/astronomy",
    "skills": [
        {"id": "next_visible_event", "name": "Next visible event", "description": "..."},
    ],
}


**Sora 2 video generation — async submit-then-poll job** (Lab 27). Modeled on OpenAI's public video-generation API pattern; not confirmed on Azure OpenAI.

In [57]:
import sys
sys.path.insert(0, "..")
import time
from azure_config import config
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")
client = OpenAI(base_url=config.openai_endpoint, api_key=token_provider)
video_deployment = config.video_deployment

try:
    video_job = client.videos.create(model=video_deployment, prompt="...", size="1280x720", seconds="4")

    while video_job.status in ("queued", "processing", "in_progress"):
        time.sleep(5)
        video_job = client.videos.retrieve(video_job.id)

    content = client.videos.download_content(video_job.id, variant="video")
    content.write_to_file("generated_video.mp4")
except Exception as ex:
    print(f"(Expected in this preview lab if Sora isn't available on your resource, or if the real API shape differs: {ex})")


(Expected in this preview lab if Sora isn't available on your resource, or if the real API shape differs: Error code: 401 - {'error': {'code': 'PermissionDenied', 'message': 'Principal does not have access to API/Operation.'}})


## 15. Image generation & editing (Lab 22)

**Generate an image from a text prompt**

In [58]:
import sys
sys.path.insert(0, "..")
import base64
from pathlib import Path
from azure_config import config
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")
client = OpenAI(base_url=config.image_endpoint, api_key=token_provider)
image_deployment = config.image_deployment

response = client.images.generate(
    model=image_deployment,
    prompt="A professional product photo of a blue ceramic coffee mug on a wooden table.",
    n=1,
    size="1024x1024",
    quality="medium",
    output_format="png",
)
image_bytes = base64.b64decode(response.data[0].b64_json)
Path("generated_image.png").write_bytes(image_bytes)
print("Saved generated_image.png")


AuthenticationError: Error code: 401 - {'error': {'code': 'PermissionDenied', 'message': 'Principal does not have access to API/Operation.'}}

**Edit an existing image with a text instruction** (whole image may change)

In [59]:
# Meant to run right after the "generate an image" cell above - edits the
# generated_image.png it just produced (reuses `client` / `image_deployment`).
with open("generated_image.png", "rb") as image_file:
    response = client.images.edit(
        model=image_deployment,
        image=image_file,
        prompt="Make the lighting warmer and add a soft bokeh background.",
        size="1024x1024",
        n=1,
    )
image_bytes = base64.b64decode(response.data[0].b64_json)
Path("edited_image.png").write_bytes(image_bytes)
print("Saved edited_image.png")


FileNotFoundError: [Errno 2] No such file or directory: 'generated_image.png'

**Masked edit / inpainting** (only the masked region changes)

In [60]:
# Meant to run right after the "generate an image" cell above (reuses `client` /
# `image_deployment`). Builds a same-dimensions mask.png on the fly (a transparent
# rectangle in the center) with Pillow, rather than requiring you to supply one.
from PIL import Image, ImageDraw

mask = Image.new("RGBA", (1024, 1024), (0, 0, 0, 255))   # fully opaque = "keep unchanged"
ImageDraw.Draw(mask).rectangle([256, 256, 768, 768], fill=(0, 0, 0, 0))   # transparent = "replace this"
mask.save("mask.png")

# ⚠️ Model capability note: this repo's current image deployment (MAI-Image-2.5, a
# Microsoft-native model family distinct from OpenAI's gpt-image-1/2) does NOT
# support a separate mask= file on images.edit() - it fails with "Exactly one
# image file must be attached for edit requests." Masked inpainting requires an
# actual gpt-image-1/2 deployment.
try:
    with open("generated_image.png", "rb") as image_file, open("mask.png", "rb") as mask_file:
        response = client.images.edit(
            model=image_deployment,
            image=image_file,
            mask=mask_file,   # transparent pixels mark the region to replace
            prompt="Replace the background with a bright blue sky.",
            size="1024x1024",
            n=1,
        )
    image_bytes = base64.b64decode(response.data[0].b64_json)
    Path("masked_edit.png").write_bytes(image_bytes)
    print("Saved masked_edit.png")
except Exception as ex:
    print(f"(Expected on a non-gpt-image deployment: {ex})")


(Expected on a non-gpt-image deployment: [Errno 2] No such file or directory: 'generated_image.png')


## 16. Content Safety, Content Understanding & Document Intelligence (Lab 23–25)

**`ContentSafetyClient` — moderate text**, one severity score (0–7) per harm category, not a single pass/fail flag.

In [61]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.ai.contentsafety import ContentSafetyClient
from azure.ai.contentsafety.models import AnalyzeTextOptions
from azure.core.credentials import AzureKeyCredential

endpoint = config.content_safety_endpoint
key = config.content_safety_key
text = "If you don't fix this immediately I will destroy your whole team."

client = ContentSafetyClient(endpoint=endpoint, credential=AzureKeyCredential(key))

result = client.analyze_text(AnalyzeTextOptions(text=text))
for category_result in result.categories_analysis:
    print(category_result.category, category_result.severity)


Hate 0
SelfHarm 0
Sexual 0
Violence 2


**`ContentSafetyClient` — moderate an image** (same client, `AnalyzeImageOptions` + raw bytes).

In [62]:
# Meant to run right after the cell above (reuses its `client`). Builds a tiny
# solid-color PNG in memory with Pillow, rather than requiring an external image file.
import io
from PIL import Image
from azure.ai.contentsafety.models import AnalyzeImageOptions, ImageData

buf = io.BytesIO()
Image.new("RGB", (64, 64), (120, 180, 220)).save(buf, format="PNG")
image_bytes = buf.getvalue()

result = client.analyze_image(AnalyzeImageOptions(image=ImageData(content=image_bytes)))
for category_result in result.categories_analysis:
    print(category_result.category, category_result.severity)


Hate 0
SelfHarm 0
Sexual 0
Violence 0


**`ContentUnderstandingClient`** ⚠️ (unverified/preview — package not installed in this repo; two Section Code files call this with different method names for the same operation) — Lab 24.

In [63]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.core.credentials import AzureKeyCredential

endpoint = config.content_understanding_endpoint   # raises MissingCredentialError until AZURE_CONTENT_UNDERSTANDING_ENDPOINT is set
key = config.content_understanding_key
document_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"

try:
    from azure.ai.contentunderstanding import ContentUnderstandingClient
    from azure.ai.contentunderstanding.models import AnalysisInput

    client = ContentUnderstandingClient(endpoint=endpoint, credential=AzureKeyCredential(key))
    poller = client.begin_analyze(analyzer_id="prebuilt-invoice", inputs=[AnalysisInput(url=document_url)])
    result = poller.result()
    print(result.as_dict())
except ImportError as ex:
    print(f"⚠️  azure-ai-contentunderstanding not installed ({ex}) - `pip install azure-ai-contentunderstanding` to try this cell for real (preview package, surface may have moved since this was written).")


MissingCredentialError: 'AZURE_CONTENT_UNDERSTANDING_ENDPOINT' is not set. Add it to the repo-root .env file (see 14-azure-ai103/README.md for the full variable list).

**`DocumentIntelligenceClient`** (proven, stable SDK) — extracts text/tables from a document as a long-running operation — Lab 25.

In [64]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential

endpoint = config.document_intelligence_endpoint   # raises MissingCredentialError until AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT is set (no resource provisioned yet)
key = config.document_intelligence_key
document_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"

client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key), api_version="2024-11-30")

poller = client.begin_analyze_document(
    model_id="prebuilt-layout",
    body=AnalyzeDocumentRequest(url_source=document_url),
    output_content_format="markdown",
)
result = poller.result()
print(result.content)   # extracted text as markdown
for table in result.tables:
    print(table.row_count, table.column_count)


MissingCredentialError: 'AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT' is not set. Add it to the repo-root .env file (see 14-azure-ai103/README.md for the full variable list).

## 17. Azure AI Search — bring-your-own-retrieval RAG (Lab 26)

**Hybrid keyword + vector search**, merged server-side via Reciprocal Rank Fusion (RRF).

In [65]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

search_endpoint = config.search_endpoint
index_name = config.search_index_name
question = "What products does CloudXeus sell?"

search_client = SearchClient(endpoint=search_endpoint, index_name=index_name, credential=DefaultAzureCredential())

results = search_client.search(
    search_text=question,                                                  # keyword side
    vector_queries=[VectorizableTextQuery(text=question, k_nearest_neighbors=3, fields="text_vector")],  # vector side
    select=["chunk", "title"],
    top=3,
)
sources = "\n\n".join(f"Source ({r['title']}):\n{r['chunk']}" for r in results)
print(sources)


Ran into a deserialization error. Ignoring since this is failsafe deserialization
Traceback (most recent call last):
  File "/Users/soubhik/AI/full-stack-ai-with-python/venv/lib/python3.11/site-packages/azure/search/documents/_utils/model_base.py", line 1051, in _failsafe_deserialize
    return _deserialize(deserializer, response.json(), module, rf, format)
                                      ^^^^^^^^^^^^^^^
  File "/Users/soubhik/AI/full-stack-ai-with-python/venv/lib/python3.11/site-packages/azure/core/rest/_http_response_impl.py", line 331, in json
    self._json = loads(self.text())
                 ^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python3.11/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python3.11/json/decoder.py", line 337, in decode
    o

HttpResponseError: Operation returned an invalid status 'Forbidden'

# Meant to run right after the cell above (reuses its `sources` / `question`).
from azure.ai.projects import AIProjectClient

project_client = AIProjectClient(endpoint=config.project_endpoint, credential=DefaultAzureCredential())
openai_client = project_client.get_openai_client()
model_deployment = config.model_deployment

prompt = f"""Answer using ONLY the sources below. If they don't contain the answer, say you don't know.

{sources}

Question: {question}
Answer:"""

response = openai_client.responses.create(model=model_deployment, input=prompt)
print(response.output_text)


In [66]:
prompt = f"""Answer using ONLY the sources below. If they don't contain the answer, say you don't know.

{sources}

Question: {question}
Answer:"""

response = openai_client.responses.create(model=model_deployment, input=prompt)


NameError: name 'sources' is not defined

## 18. Translation — LLM-prompted vs. dedicated service (Lab 28)

**LLM-prompted translation** — flexible instructions, one call per target language.

In [67]:
import sys
sys.path.insert(0, "..")
from azure_config import config
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(endpoint=config.project_endpoint, credential=DefaultAzureCredential())
openai_client = project_client.get_openai_client()
model_deployment = config.model_deployment
target_language = "French"
text = "Could you please confirm my reservation for tomorrow evening?"

response = openai_client.responses.create(
    model=model_deployment,
    input=[
        {"type": "message", "role": "system", "content": f"You are a professional translator. Translate into {target_language}. Reply with ONLY the translated text."},
        {"type": "message", "role": "user", "content": text},
    ],
)
translated = response.output_text
print(translated)


Pourriez-vous confirmer ma réservation pour demain soir ?


**Azure AI Translator — dedicated REST service**, one call can target MANY languages at once (no Python SDK package in common use — plain `requests`).

In [68]:
import sys
sys.path.insert(0, "..")
import requests
from azure_config import config

endpoint = config.translator_endpoint   # raises MissingCredentialError until AZURE_TRANSLATOR_ENDPOINT (or AZURE_LANGUAGE_ENDPOINT) is set
key = config.translator_key
text = "Could you please confirm my reservation for tomorrow evening?"

headers = {"Ocp-Apim-Subscription-Key": key, "Content-Type": "application/json"}
url = f"{endpoint}translator/text/translate?api-version=2025-10-01-preview"
body = {"inputs": [{"Text": text, "language": "en", "targets": [{"language": "fr"}, {"language": "ja"}]}]}

response = requests.post(url, headers=headers, json=body)
response.raise_for_status()
for t in response.json()["value"][0]["translations"]:
    print(t["language"], t["text"])


MissingCredentialError: 'AZURE_LANGUAGE_ENDPOINT' is not set. Add it to the repo-root .env file (see 14-azure-ai103/README.md for the full variable list).